# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Josh Soussan  Student ID: B00821791
2. Full name: Edouard Souied  Student ID: B00825107
3. Full name: Nicolas Salapete  Student ID: B00821820

**Group / repo name:** `aidams-lab4-Soussan-Souied-Salapete`
**Submitter (one person):** Nicolas Salapete
**Repo URL:**

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).

## Setup (given)

In [1]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA = Path("data")
excerpts = pd.DataFrame(json.loads((DATA / "excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads((DATA / "gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads((DATA / "recorded_responses.json").read_text()))
corpus = json.loads((DATA / "aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [2]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [3]:
excerpt_text = excerpts.set_index("id")["text"]
Q1_IDS = ["E03", "E09", "E19"]

for excerpt_id in Q1_IDS:
    print(f"{excerpt_id}: {excerpt_text[excerpt_id]}\n")

# Our own labels, written by reading the three excerpts and the rules in PROMPT, before opening `gold`.
#  - E03 gives its emissions "in units of 10,000 tonnes", so 4,108 means 41,080,000 t.
#  - E09 only gives Scope 1 + 2 together, and the prompt says null if a scope is not reported separately.
#  - E19 gives production in short tons (1 short ton = 0.907185 metric tonnes).
SHORT_TON_IN_TONNES = 0.907185

hand_labels = pd.DataFrame([
    {"id": "E03", "company": "Lianhe Iron & Steel Group", "report_year": 2023,
     "scope1_tco2": 4_108 * 10_000, "scope2_tco2": 512 * 10_000, "production_mt": 21.4, "net_zero_year": 2060},
    {"id": "E09", "company": "Ostend Metals NV", "report_year": 2024,
     "scope1_tco2": None, "scope2_tco2": None, "production_mt": 4.4, "net_zero_year": 2050},
    {"id": "E19", "company": "Bluewater Iron Corp.", "report_year": 2024,
     "scope1_tco2": 5_100_000, "scope2_tco2": 410_000, "production_mt": round(3.3 * SHORT_TON_IN_TONNES, 2),
     "net_zero_year": 2050},
]).set_index("id")


def same_as_gold(ours, truth):
    """Quick check for Q1 only: both empty, same name (ignoring case and suffixes), or numbers within 2%."""
    if pd.isna(ours) or pd.isna(truth):
        return bool(pd.isna(ours) and pd.isna(truth))
    if isinstance(truth, str):
        return truth.lower() in str(ours).lower()
    return abs(ours - truth) <= 0.02 * abs(truth)


q1_rows = []
for excerpt_id in Q1_IDS:
    for field in FIELDS:
        ours, truth = hand_labels.at[excerpt_id, field], gold.at[excerpt_id, field]
        q1_rows.append({"id": excerpt_id, "field": field, "ours": ours, "gold": truth,
                        "same": same_as_gold(ours, truth)})
q1_check = pd.DataFrame(q1_rows)
display(q1_check)
print(f"Our labels agree with gold on {q1_check.same.sum()} of {len(q1_check)} fields.")

E03: Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

E09: Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050.

E19: Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.



,id,field,ours,gold,same
0,E03,company,Lianhe Iron & Steel Group,Lianhe Iron & Steel Group,True
1,E03,report_year,2023,2023,True
2,E03,scope1_tco2,"41,080,000.000","41,080,000.000",True
3,E03,scope2_tco2,"5,120,000.000","5,120,000.000",True
4,E03,production_mt,21.400,21.400,True
5,E03,net_zero_year,2060,2060,True
6,E09,company,Ostend Metals NV,Ostend Metals,True
7,E09,report_year,2024,2024,True
8,E09,scope1_tco2,NaN,NaN,True
9,E09,scope2_tco2,NaN,NaN,True


Our labels agree with gold on 18 of 18 fields.


**Your answer (Q1):**

Our labels (table above) end up matching the gold on all 18 fields, but three of them needed real thought:

* **Hardest field: `production_mt` for E19.** It is the only value that needs knowledge that is not in the text: 3.3 million *short* tons is 3.3 × 0.907 = **2.99 Mt**, not 3.3. The sentence also starts with "Shipments and production", so you have to check that 3.3 is production and not shipments.
* **E03** looks easy until you notice "(unit: 10,000 tonnes CO2)" in the middle of the sentence: 4,108 means **41.08 MtCO2**. It is very easy to miss when reading quickly.
* **E09** is a judgment call. The report only gives 8.7 MtCO2e for Scope 1 and 2 together. The prompt says null when a scope is not reported separately, so both stay empty, but putting 8.7 Mt in Scope 1 as a "best guess" is tempting.
* Even the company name is a convention: we wrote "Bluewater Iron Corp." and "Ostend Metals NV", while the gold drops the legal suffix. That is why B1 uses a containment match for names.

If two of us disagreed on fields like E09 Scope 1 or the E19 conversion, it would show that the "truth" is partly a set of labelling rules, not a fact sitting in the text. The gold labels have their own error rate, so what Part B measures is really "agreement with our rule book". Ambiguous fields need written rules and ideally two annotators, and a small accuracy gap between two models should not be over-read.

## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [4]:
from pydantic import Field


class SteelDisclosure(BaseModel):
    """One extracted record. The keys are required: a key the model forgets is a failure, not a silent null."""
    company: str
    report_year: int
    scope1_tco2: Optional[float]
    scope2_tco2: Optional[float]
    production_mt: Optional[float]
    net_zero_year: Optional[int]
    confidence: float = Field(ge=0, le=1)


# From the first "{" to the last "}", across line breaks: this drops "Sure! Here is..." and ``` fences.
JSON_OBJECT = re.compile(r"\{.*\}", flags=re.S)


def parse_response(raw):
    """Find the JSON object inside the raw text and validate it.
    Returns (record, None) on success and (None, error_message) on failure."""
    match = JSON_OBJECT.search(raw)
    if match is None:
        return None, "no JSON object in the response"
    try:
        return SteelDisclosure.model_validate_json(match.group(0)), None
    except ValidationError as error:
        problem = error.errors()[0]          # the first problem is enough to understand what went wrong
        where = ".".join(str(part) for part in problem["loc"])
        return None, f"{where}: {problem['msg']} (got {problem['input']!r})"


parsed_rows, failures = [], []
for response in recorded.itertuples():
    record, error_message = parse_response(response.raw_text)
    if record is None:
        failures.append({"id": response.id, "kind": response.kind, "sample": response.sample,
                         "error": error_message, "raw_text": response.raw_text})
    else:
        parsed_rows.append({"id": response.id, "kind": response.kind, "sample": response.sample,
                            **record.model_dump()})

# Years as nullable integers, like in `gold`, so 2050 does not turn into 2050.0 next to a missing value
pred = pd.DataFrame(parsed_rows).astype({"report_year": "Int64", "net_zero_year": "Int64"})
print(f"Parsed {len(pred)} of {len(recorded)} responses; {len(failures)} failed.")
display(pred.head())

Parsed 119 of 120 responses; 1 failed.


,id,kind,sample,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year,confidence
0,E01,greedy,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,2050,0.910
1,E01,sample,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,2050,0.940
2,E01,sample,1,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,2050,0.910
3,E01,sample,2,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,2050,0.930
4,E01,sample,3,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,2050,0.960


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [5]:
# (a) The failures and their raw text
for failure in failures:
    print(f"FAILED  {failure['id']} ({failure['kind']}, sample {failure['sample']}): {failure['error']}")
    print(failure["raw_text"], "\n")

# A response "needs the regex" when it is not a bare JSON object: there is text or a code fence around it.
raw_stripped = recorded.raw_text.str.strip()
needs_regex = ~(raw_stripped.str.startswith("{") & raw_stripped.str.endswith("}"))
print(f"Responses wrapped in text or code fences: {needs_regex.sum()} of {len(recorded)}")
display(recorded.loc[needs_regex, ["id", "kind", "sample"]].reset_index(drop=True).T)
print("Example of a wrapped response:\n" + recorded.loc[needs_regex, "raw_text"].iloc[0])

FAILED  E11 (greedy, sample 0): production_mt: Input should be a valid number, unable to parse string as a number (got '8.1 Mt')
{
  "company": "Anatolia Celik",
  "report_year": 2024,
  "scope1_tco2": 4120000,
  "scope2_tco2": 2350000,
  "production_mt": "8.1 Mt",
  "net_zero_year": 2053,
  "confidence": 0.94
} 

Responses wrapped in text or code fences: 7 of 120


,0,1,2,3,4,5,6
id,E02,E04,E07,E09,E10,E14,E17
kind,sample,greedy,sample,sample,sample,sample,sample
sample,4,0,1,1,2,3,3


Example of a wrapped response:
Sure! Here is the extracted data:
```json
{
  "company": "Kestrel Metals",
  "report_year": 2024,
  "scope1_tco2": 5.9,
  "scope2_tco2": 210000,
  "production_mt": 3.1,
  "net_zero_year": 2045,
  "confidence": 0.95
}
```


**Your answer (Q2):**

(a) **1 of the 120 responses fails to parse**: the greedy answer for **E11**, where the model wrote `"production_mt": "8.1 Mt"`. That is a string with a unit inside, so pydantic cannot read it as a number. The value is also wrong in content: 8.1 Mt is the company's *shipments*, while production is 9.8 Mt. **7 responses** (6%) were wrapped in text ("Sure! Here is the extracted data:" plus a ```` ```json ```` fence) and only parsed thanks to the regex: E04 greedy and one sample each of E02, E07, E09, E10, E14 and E17.

(b) Two ways to handle a failed response in production:
* **Ask the model again**, ideally with the error message in the new prompt ("production_mt must be a number"). *Risk:* the model fixes the format but not the content. Here it would most likely answer 8.1, a valid number that is wrong (shipments), and the error becomes invisible. Retries also cost time and money.
* **Send the excerpt to a human**, and store the record as "missing, to review" until then. *Risk:* analyst time, and if the queue is not worked through, the company silently drops out of the database, which biases totals downwards (this is exactly what E11 does to the sector total in B3).

(A third option, cleaning the string automatically, "8.1 Mt" → 8.1, is the worst: it accepts a wrong value with no flag at all.)

Counting a failure as wrong on every field is the honest choice because the pipeline really produced nothing usable for that excerpt: for the user, it is six missing values. If we dropped failures from the denominator, we would report the accuracy of the responses that happened to parse, a selected and probably easier subset, and accuracy would go *up* because the model failed. Failures also tend to happen on the hard excerpts (E11 has a production versus shipments trap), so leaving them out flatters the model twice.

> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [6]:
# Optional. This only runs if BACKEND and MODEL (in the ask_llm cell) point to a real model.
# We had no API key and no local Ollama server, so in our run the cell just says so.
if BACKEND == "recorded":
    print("BACKEND is 'recorded': no live model available, A2 skipped.")
else:
    recorded_greedy = pred[pred.kind == "greedy"].set_index("id")
    live_rows = []
    for excerpt_id in Q1_IDS:
        raw_answer = ask_llm(PROMPT.format(text=excerpt_text[excerpt_id]), temperature=0.0)
        live_record, live_error = parse_response(raw_answer)
        for field in FIELDS:
            live_rows.append({
                "id": excerpt_id, "field": field,
                "our model": getattr(live_record, field) if live_record else f"parse failure: {live_error}",
                "recorded greedy": recorded_greedy[field].get(excerpt_id),
                "gold": gold.at[excerpt_id, field]})
    display(pd.DataFrame(live_rows))

BACKEND is 'recorded': no live model available, A2 skipped.


---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [7]:
NUMBER_FIELDS = ["scope1_tco2", "scope2_tco2", "production_mt"]
YEAR_FIELDS = ["report_year", "net_zero_year"]
RELATIVE_TOLERANCE = 0.02


def clean_name(name):
    """Lower case, punctuation turned into spaces, single spaces: 'Kestrel Metals plc.' -> 'kestrel metals plc'."""
    return " ".join(re.sub(r"[^\w\s]", " ", str(name).lower()).split())


def field_correct(field, pred_value, gold_value):
    """Is one extracted value right? Rules of B1: missing vs missing, name containment, exact year, 2% on numbers."""
    pred_missing, gold_missing = bool(pd.isna(pred_value)), bool(pd.isna(gold_value))
    if pred_missing or gold_missing:
        return pred_missing and gold_missing     # both empty is right, only one empty is wrong
    if field == "company":
        pred_name, gold_name = clean_name(pred_value), clean_name(gold_value)
        return pred_name != "" and (pred_name in gold_name or gold_name in pred_name)
    if field in YEAR_FIELDS:
        return int(pred_value) == int(gold_value)
    return abs(float(pred_value) - float(gold_value)) <= RELATIVE_TOLERANCE * abs(float(gold_value))


def score(predictions):
    """One row per (excerpt, field) with pred, gold and correct.
    `predictions` holds at most one parsed response per excerpt. An excerpt whose response failed to
    parse has no row there, so after the reindex it is scored wrong on every field."""
    by_excerpt = predictions.assign(parsed=True).set_index("id").reindex(gold.index)
    parsed = by_excerpt["parsed"].eq(True)
    rows = []
    for excerpt_id in gold.index:
        for field in FIELDS:
            pred_value, gold_value = by_excerpt.at[excerpt_id, field], gold.at[excerpt_id, field]
            is_correct = bool(parsed[excerpt_id]) and field_correct(field, pred_value, gold_value)
            rows.append({"id": excerpt_id, "field": field, "pred": pred_value, "gold": gold_value,
                         "correct": is_correct})
    return pd.DataFrame(rows)


greedy_pred = pred[pred.kind == "greedy"]
greedy_scores = score(greedy_pred)

field_accuracy = (greedy_scores.groupby("field", sort=False).correct
                  .agg(correct="sum", fields="count", accuracy="mean"))
display(field_accuracy)
print(f"Overall greedy accuracy: {greedy_scores.correct.mean():.1%} "
      f"({greedy_scores.correct.sum()} of {len(greedy_scores)} fields)")

,correct,fields,accuracy
field,,,
company,19,20,0.950
report_year,18,20,0.900
scope1_tco2,15,20,0.750
scope2_tco2,17,20,0.850
production_mt,18,20,0.900
net_zero_year,17,20,0.850


Overall greedy accuracy: 86.7% (104 of 120 fields)


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [8]:
overall_accuracy = greedy_scores.correct.mean()
worst_field = field_accuracy.accuracy.idxmin()
print(f"(a) Overall accuracy {overall_accuracy:.1%}; worst field: {worst_field} "
      f"({field_accuracy.at[worst_field, 'accuracy']:.0%})\n")

# (b) We read every excerpt with an error and gave each wrong field a type by hand.
ERROR_TYPES = {
    ("E03", "scope1_tco2"): "unit conversion",          # "unit: 10,000 tonnes" not applied
    ("E03", "scope2_tco2"): "unit conversion",
    ("E19", "production_mt"): "unit conversion",        # short tons kept as metric tonnes
    ("E07", "scope1_tco2"): "wrong year in a table",    # took the 2023 column
    ("E08", "report_year"): "fiscal year rule",         # FY Apr 2024 - Mar 2025 should be 2024
    ("E09", "scope1_tco2"): "wrong scope",              # Scope 1+2 combined put into Scope 1
    ("E05", "scope1_tco2"): "invented value",           # intensity x production, never reported
    ("E10", "net_zero_year"): "invented value",         # 2030 is an intensity target, not net zero
    ("E18", "scope2_tco2"): "invented value",           # "will be disclosed from 2025" became 0
    ("E20", "net_zero_year"): "invented value",         # 2026 is the first-coil date
}
failed_greedy_ids = {failure["id"] for failure in failures if failure["kind"] == "greedy"}

greedy_errors = greedy_scores[~greedy_scores.correct].copy()
greedy_errors["error_type"] = [
    "parse failure" if row.id in failed_greedy_ids else ERROR_TYPES.get((row.id, row.field), "other")
    for row in greedy_errors.itertuples()]
display(greedy_errors)

print("Wrong fields by error type:")
print(greedy_errors.error_type.value_counts().to_string())
print("\nDistinct mistakes (a parse failure counted once):")
print(greedy_errors.drop_duplicates(["id", "error_type"]).error_type.value_counts().to_string())

print("\nExcerpts with at least one error:")
for excerpt_id in greedy_errors.id.unique():
    print(f"  {excerpt_id}: {excerpt_text[excerpt_id]}")

(a) Overall accuracy 86.7%; worst field: scope1_tco2 (75%)



,id,field,pred,gold,correct,error_type
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000",False,unit conversion
15,E03,scope2_tco2,"512,000.000","5,120,000.000",False,unit conversion
26,E05,scope1_tco2,"42,042,000.000",NaN,False,invented value
38,E07,scope1_tco2,"14,200,000.000","13,800,000.000",False,wrong year in a table
43,E08,report_year,2025,2024,False,fiscal year rule
50,E09,scope1_tco2,"8,700,000.000",NaN,False,wrong scope
59,E10,net_zero_year,2030,<NA>,False,invented value
60,E11,company,NaN,Anatolia Celik,False,parse failure
61,E11,report_year,<NA>,2024,False,parse failure
62,E11,scope1_tco2,NaN,"4,120,000.000",False,parse failure


Wrong fields by error type:
error_type
parse failure            6
invented value           4
unit conversion          3
wrong year in a table    1
fiscal year rule         1
wrong scope              1

Distinct mistakes (a parse failure counted once):
error_type
invented value           4
unit conversion          2
wrong year in a table    1
fiscal year rule         1
wrong scope              1
parse failure            1

Excerpts with at least one error:
  E03: Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.
  E05: Tamarind Steel Ltd - Business Responsibility and Sustainability Report FY2024. Crude steel production rose 6% to 18.2 million tonnes. Our CO

**Your answer (Q3):**

(a) Overall greedy accuracy is **86.7% (104 of 120 fields)**. The worst field is **`scope1_tco2`: 15 of 20 right (75%)**, followed by Scope 2 and the net zero year (85% each).

(b) We read the excerpt behind each of the 16 wrong fields:

| Error type | Wrong fields | Where |
|---|---|---|
| Parse failure | 6 | E11: one bad response, so all six fields are lost |
| Value invented for a missing field | 4 | E05 Scope 1 = 2.31 × 18.2 Mt, computed from the intensity; E10 net zero "2030" (an intensity target); E18 Scope 2 = 0 ("will be disclosed from 2025"); E20 net zero "2026" (the first-coil date) |
| Unit conversion | 3 | E03 Scope 1 and 2 (the 10,000-tonne unit ignored, 10 times too small); E19 production (short tons kept as tonnes) |
| Wrong year in a table | 1 | E07 Scope 1: 14.2 Mt from the 2023 column instead of 13.8 |
| Fiscal year | 1 | E08: FY April 2024 to March 2025 should be 2024, the model took the report's title year |
| Wrong scope | 1 | E09: the combined Scope 1 + 2 figure put into Scope 1 |

Counted in fields, the parse failure comes first, but it is a single mistake. Counted in distinct mistakes, **inventing a value where the report has none is the most frequent type (4 of 10)**, even though the prompt says in capital letters "Never compute a value that is not reported". Unit conversion comes second and does the biggest damage in size (E03 is off by a factor of 10).

(c) The errors we could easily make ourselves are the "reading" ones: the 10,000-tonne unit of E03 and the short tons of E19 (exactly the traps we met in Q1), the fiscal year of E08 (the report is titled 2025) and picking the wrong column in E07. We would be much less likely to invent values: a human does not multiply an intensity by production to fill a Scope 1 cell, or write 0 for "not disclosed yet". Those errors are specific to the model, and they are the more dangerous ones because the output looks complete.

## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [9]:
# 1. Bayesian: uniform Beta(1, 1) prior, k successes out of n fields
k = int(greedy_scores.correct.sum())
n = len(greedy_scores)
posterior = stats.beta(1 + k, 1 + n - k)
beta_ci = posterior.ppf([0.05, 0.95])

# 2. Bootstrap over excerpts: the six fields of one excerpt stay together in every resample
per_excerpt = greedy_scores.groupby("id").correct.agg(["sum", "count"])
correct_per_excerpt = per_excerpt["sum"].to_numpy()
fields_per_excerpt = per_excerpt["count"].to_numpy()
n_excerpts = len(per_excerpt)

N_BOOT = 2_000
rng = np.random.default_rng(SEED)
resampled = rng.integers(0, n_excerpts, size=(N_BOOT, n_excerpts))
boot_accuracy = correct_per_excerpt[resampled].sum(axis=1) / fields_per_excerpt[resampled].sum(axis=1)
boot_ci = np.percentile(boot_accuracy, [5, 95])

print(f"k = {k} correct out of n = {n} fields ({k / n:.1%})")
print(f"Beta 90% interval:      [{beta_ci[0]:.3f}, {beta_ci[1]:.3f}], width {beta_ci[1] - beta_ci[0]:.3f}")
print(f"Bootstrap 90% interval: [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}], width {boot_ci[1] - boot_ci[0]:.3f}")

# For Q4 (b): how does the bootstrap width shrink if we had labelled more excerpts like these ones?
print("\nBootstrap width with more excerpts of the same kind:")
for n_labelled in [20, 40, 80, 160]:
    bigger_sample = np.random.default_rng(SEED).integers(0, n_excerpts, size=(N_BOOT, n_labelled))
    accuracy_draws = (correct_per_excerpt[bigger_sample].sum(axis=1)
                      / fields_per_excerpt[bigger_sample].sum(axis=1))
    low, high = np.percentile(accuracy_draws, [5, 95])
    print(f"  {n_labelled:>3} excerpts: width {high - low:.3f}")

# For Q4 (c): how plausible is the vendor's 95%?
print(f"\nP(true accuracy >= 95%) under the Beta posterior: {posterior.sf(0.95):.4f}")
print(f"Share of bootstrap resamples with accuracy >= 95%: {(boot_accuracy >= 0.95).mean():.4f}")
print(f"Per-excerpt errors: {(fields_per_excerpt - correct_per_excerpt).tolist()}")

k = 104 correct out of n = 120 fields (86.7%)
Beta 90% interval:      [0.806, 0.908], width 0.102
Bootstrap 90% interval: [0.775, 0.933], width 0.158

Bootstrap width with more excerpts of the same kind:


   20 excerpts: width 0.158
   40 excerpts: width 0.112
   80 excerpts: width 0.081
  160 excerpts: width 0.055

P(true accuracy >= 95%) under the Beta posterior: 0.0001
Share of bootstrap resamples with accuracy >= 95%: 0.0225
Per-excerpt errors: [0, 0, 2, 0, 1, 0, 1, 1, 1, 1, 6, 0, 0, 0, 0, 0, 0, 1, 1, 1]


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

(a) **Beta-Binomial: [0.806, 0.908]** (width 0.102). **Bootstrap over excerpts: [0.775, 0.933]** (width 0.158), about 1.5 times wider.

The Beta interval treats the 120 fields as 120 independent coin flips. They are not: errors come in clusters inside one excerpt. E11 alone gives 6 errors (the whole response failed) and E03 gives 2, while 10 excerpts have none. Resampling excerpts keeps these clusters together, so a resample that draws E11 twice, or not at all, moves the accuracy a lot. In practice we have about 20 independent observations, not 120, and the bootstrap interval is the honest one.

(b) The width shrinks like 1/√n, so halving it takes about **4 times as many excerpts: around 80 instead of 20**, i.e. 60 more to label. Our resampling check agrees: width 0.158 with 20 excerpts, 0.112 with 40 and **0.081 with 80**.

(c) **No.** 95% is above the top of both intervals. Under the Beta posterior the probability that the true accuracy is 95% or more is 0.0001, and only 2% of the bootstrap resamples reach it. Even if we scored E11 on its content instead of as a full failure, accuracy would be about 91% (109 of 120), still below. Two caveats: our 20 excerpts were written to contain traps, so they may be harder than the vendor's test set, and a single "95%" with no interval and no test set cannot really be checked.

## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [10]:
greedy_by_excerpt = greedy_pred.set_index("id").reindex(gold.index)   # E11 (parse failure) becomes empty

downstream = pd.DataFrame({
    "scope1_llm": greedy_by_excerpt["scope1_tco2"],
    "scope1_gold": gold["scope1_tco2"],
})
# For the sector total a missing value counts as 0, on both sides
downstream["error_t"] = downstream.scope1_llm.fillna(0) - downstream.scope1_gold.fillna(0)

TONNES_PER_MT = 1e6
total_llm = downstream.scope1_llm.sum() / TONNES_PER_MT       # .sum() skips NaN, i.e. treats it as 0
total_gold = downstream.scope1_gold.sum() / TONNES_PER_MT
net_error = downstream.error_t.sum() / TONNES_PER_MT
absolute_error = downstream.error_t.abs().sum() / TONNES_PER_MT

# Emission intensity (tCO2 per tonne of steel) only where Scope 1 and production are both present
intensity_llm = (greedy_by_excerpt["scope1_tco2"] / (greedy_by_excerpt["production_mt"] * TONNES_PER_MT)).dropna()
intensity_gold = (gold["scope1_tco2"] / (gold["production_mt"] * TONNES_PER_MT)).dropna()

print(f"Sector Scope 1 total: LLM {total_llm:,.2f} MtCO2 vs gold {total_gold:,.2f} MtCO2")
print(f"Net error: {net_error:+,.2f} MtCO2 ({net_error / total_gold:+.1%} of the gold total)")
print(f"Absolute error: {absolute_error:,.2f} MtCO2 ({absolute_error / total_gold:.1%} of the gold total)")
print(f"Median intensity: LLM {intensity_llm.median():.3f} t/t ({len(intensity_llm)} companies) "
      f"vs gold {intensity_gold.median():.3f} t/t ({len(intensity_gold)} companies)")

print("\nExcerpts with a Scope 1 error, biggest first (MtCO2):")
scope1_errors = downstream[downstream.error_t != 0].copy()
scope1_errors["error_mt"] = scope1_errors.error_t / TONNES_PER_MT
scope1_errors["share_of_absolute_error"] = scope1_errors.error_t.abs() / downstream.error_t.abs().sum()
display(scope1_errors.sort_values("error_mt", key=abs, ascending=False))

Sector Scope 1 total: LLM 252.11 MtCO2 vs gold 242.06 MtCO2
Net error: +10.05 MtCO2 (+4.2% of the gold total)
Absolute error: 92.23 MtCO2 (38.1% of the gold total)
Median intensity: LLM 1.895 t/t (18 companies) vs gold 1.865 t/t (17 companies)

Excerpts with a Scope 1 error, biggest first (MtCO2):


,scope1_llm,scope1_gold,error_t,error_mt,share_of_absolute_error
id,,,,,
E05,"42,042,000.000",NaN,"42,042,000.000",42.042,0.456
E03,"4,108,000.000","41,080,000.000","-36,972,000.000",-36.972,0.401
E09,"8,700,000.000",NaN,"8,700,000.000",8.700,0.094
E11,NaN,"4,120,000.000","-4,120,000.000",-4.120,0.045
E07,"14,200,000.000","13,800,000.000","400,000.000",0.400,0.004


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

(a) Sector Scope 1 total: **LLM 252.1 MtCO2 against 242.1 MtCO2 in the gold**. The **net error is +10.1 MtCO2 (+4.2%)**, but the **absolute error is 92.2 MtCO2 (38% of the total)**. Two excerpts make 86% of the absolute error:
* **E05: +42.0 Mt**, a Scope 1 the model made up by multiplying the intensity by production;
* **E03: −37.0 Mt**, the 10,000-tonne unit ignored.

They have opposite signs and almost cancel, which is why the net error looks small. That is luck, not quality.

(b) Field accuracy gives every field the same weight, but in a total a field weighs as much as the company's emissions. The errors are not small noise around the truth: they are rare and huge (a factor-10 unit error, an invented 42 Mt). So 87% of fields right can still leave a total off by 38% in absolute terms. This is the Session 4 message, "the mean hides the tail": the risk sits in a few big errors, not in the average. The median intensity barely moves (**1.895 vs 1.865 t/t**) because a median only looks at the middle of the ranking: E03's absurd 0.19 t/t and E05's 2.31 t/t move a couple of ranks, not the value in the middle. It also only uses companies where both numbers exist, so E11 simply drops out. The median is robust, and that robustness hides the errors that hit the total.

(c) Next to the total we would report:
* a range rather than one number, since on this test the total could be off by tens of MtCO2 either way;
* how many companies have no separate Scope 1 (E05, E09, plus the E11 failure) and how they were treated;
* which values were checked by a human, and the few companies that drive most of the total;
* the result of a plausibility check: E03's 0.19 tCO2 per tonne of steel is far below any blast-furnace producer and should have been flagged before reaching the lender.

---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [11]:
from collections import Counter

N_SAMPLES = 5
samples = pred[pred.kind == "sample"]


def answer_key(value):
    """Turn an answer into a string so None, NaN, numbers and names can be counted together."""
    if pd.isna(value):
        return json.dumps(None)
    if isinstance(value, (int, float, np.number)):
        return json.dumps(float(value))           # 2050 and 2050.0 are the same answer
    return json.dumps(str(value))


vote_rows = []
for excerpt_id in gold.index:
    excerpt_samples = samples[samples.id == excerpt_id]
    for field in FIELDS:
        answers = excerpt_samples[field].tolist()
        counts = Counter(answer_key(answer) for answer in answers)
        # most_common keeps the first-seen answer on a tie; a failed sample simply never agrees
        vote_key, n_agree = counts.most_common(1)[0] if counts else (None, 0)
        vote = next((answer for answer in answers if answer_key(answer) == vote_key), None)
        gold_value = gold.at[excerpt_id, field]
        vote_rows.append({"id": excerpt_id, "field": field, "vote": vote, "gold": gold_value,
                          "n_agree": n_agree, "agreement": n_agree / N_SAMPLES,
                          "n_distinct_answers": len(counts),
                          "tie": list(counts.values()).count(n_agree) > 1,
                          "correct": n_agree > 0 and field_correct(field, vote, gold_value)})
votes = pd.DataFrame(vote_rows)
votes["agreement_bin"] = np.select([votes.n_agree == 5, votes.n_agree == 4], ["5/5", "4/5"], "3/5 or less")

greedy_accuracy = greedy_scores.correct.mean()
vote_accuracy = votes.correct.mean()
print(f"Greedy accuracy: {greedy_accuracy:.1%} | majority-vote accuracy: {vote_accuracy:.1%}")

accuracy_by_agreement = (votes.groupby("agreement_bin").correct
                         .agg(fields="count", correct="sum", accuracy="mean")
                         .reindex(["5/5", "4/5", "3/5 or less"]))
display(accuracy_by_agreement)

print(f"Fields where two answers tie for the vote: {votes.tie.sum()}")

Greedy accuracy: 86.7% | majority-vote accuracy: 95.0%

,fields,correct,accuracy
agreement_bin,,,
5/5,93,91,0.978
4/5,15,14,0.933
3/5 or less,12,9,0.750


Fields where two answers tie for the vote: 0


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [12]:
# Beta(1, 1) 90% interval on the accuracy of each agreement bin
interval_rows = []
for bin_name, bin_votes in votes.groupby("agreement_bin"):
    n_bin, k_bin = len(bin_votes), int(bin_votes.correct.sum())
    bin_posterior = stats.beta(1 + k_bin, 1 + n_bin - k_bin)
    low, high = bin_posterior.ppf([0.05, 0.95])
    interval_rows.append({"agreement": bin_name, "fields": n_bin, "correct": k_bin, "accuracy": k_bin / n_bin,
                          "90% low": low, "90% high": high, "P(accuracy > 95%)": bin_posterior.sf(0.95)})
display(pd.DataFrame(interval_rows).set_index("agreement").reindex(["5/5", "4/5", "3/5 or less"]))

# Share of all vote errors that sit in each bin: does low agreement point at the errors?
print("Where the vote errors are:")
print(votes[~votes.correct].agreement_bin.value_counts().reindex(["5/5", "4/5", "3/5 or less"], fill_value=0).to_string())

,fields,correct,accuracy,90% low,90% high,P(accuracy > 95%)
agreement,,,,,,
5/5,93,91,0.978,0.935,0.991,0.855
4/5,15,14,0.933,0.736,0.977,0.189
3/5 or less,12,9,0.750,0.505,0.887,0.003


Where the vote errors are:
agreement_bin
5/5            2
4/5            1
3/5 or less    3


**Your answer (Q6):**

(a) **Greedy accuracy 86.7%**, **majority-vote accuracy 95.0%** (114 of 120). By agreement:

| Agreement | Fields | Accuracy |
|---|---|---|
| 5/5 | 93 | **97.8%** (91 right) |
| 4/5 | 15 | **93.3%** (14 right) |
| 3/5 or less | 12 | **75.0%** (9 right) |

(b) Yes, low agreement is a good warning sign. A field at 3/5 or less is wrong 25% of the time, against 2% for a unanimous one, more than 10 times more often. And only 27 of the 120 fields (23%) are below 5/5, so the signal focuses review effort where it is needed. But it is not a complete detector: of the 6 errors left after the vote, 4 have agreement below 5/5 and **2 are unanimous**. Disagreement catches random errors, not systematic ones (Q7).

(c) For unanimous (5/5) fields the accuracy is 97.8% with a **90% Beta interval of [0.935, 0.991]**. The lower bound is below 95%, so **we cannot claim it is above 95%** at this level of confidence. The posterior probability of being above 95% is 0.86: "likely", not "very likely". With 93 fields and 2 errors, a true accuracy of 93.5% is still compatible with the data.

## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [13]:
confidently_wrong = votes[(votes.n_agree == N_SAMPLES) & ~votes.correct]
display(confidently_wrong[["id", "field", "vote", "gold", "agreement"]])

for row in confidently_wrong.itertuples():
    print(f"{row.id} / {row.field}: all five samples say {row.vote:,}, gold is {row.gold:,}")
    print(f"   {excerpt_text[row.id]}\n")

# The greedy answer made the same two mistakes, so temperature 0 does not help either
greedy_lookup = greedy_scores.set_index(["id", "field"]).correct
print("Greedy also wrong on these fields:",
      [not greedy_lookup[(row.id, row.field)] for row in confidently_wrong.itertuples()])

,id,field,vote,gold,agreement
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000",1.000
112,E19,production_mt,3.300,2.990,1.000


E03 / scope1_tco2: all five samples say 4,108,000.0, gold is 41,080,000.0
   Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

E19 / production_mt: all five samples say 3.3, gold is 2.99
   Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.

Greedy also wrong on these fields: [True, True]


**Your answer (Q7):**

Two fields have all five samples in agreement and wrong: **E03 Scope 1** (4,108,000 t instead of 41,080,000 t) and **E19 production** (3.3 Mt instead of 2.99 Mt). The greedy answer made the same two mistakes.

(a) Both are **unit conversions with an unusual unit**: the Chinese "10,000 tonnes" unit and US short tons. The model reads the number correctly but never converts it. It makes the same mistake every time because it comes from how the model reads this kind of text, not from chance.

(b) In Session 4 terms this is a **systematic error (a bias)**, not **random noise**. Sampling at temperature 0.7 only explores the model's own randomness. Five draws from the same model with the same prompt share the same blind spot, so their errors are perfectly correlated, and an ensemble only cancels errors that are independent across its members. What would catch them:
* a **unit-conversion rule**: a deterministic check that looks for "10,000 tonnes", "short tons" and similar in the text, then converts or flags. This is the most reliable fix, because it does not depend on the model;
* a **human**, if we know where to look (see Q9);
* a **second model** only helps if it was trained differently enough not to share the blind spot, which is not guaranteed;
* a **different prompt** that lists these units would probably fix these two cases, but not the next unit nobody thought of.

(c) The analogue is the **multi-model ensemble whose members agree because they share the same assumption**: a narrow spread between models is not proof that they are right, it can just mean they all make the same structural error. We saw the same thing in our own Lab 3: every BF-BOF plant got the same emission-factor draw, so Angang came out ahead of Baoshan in 100% of the simulations, and re-drawing utilisation 1,000 times never told us whether the factor table itself was biased.

## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [14]:
from sklearn.metrics import roc_auc_score

# Score each of the 5 sample sets with the same `score` function, then ask: are all six fields right?
sample_scores = pd.concat([score(samples[samples["sample"] == sample_number]).assign(sample=sample_number)
                           for sample_number in range(N_SAMPLES)])
fully_correct = sample_scores.groupby(["id", "sample"]).correct.all().rename("fully_correct")

calibration = samples.set_index(["id", "sample"])[["confidence"]].join(fully_correct)
print(f"Sample responses: {len(calibration)}")
print(f"Mean stated confidence: {calibration.confidence.mean():.3f}")
print(f"Share with all six fields correct: {calibration.fully_correct.mean():.3f}")

calibration["confidence_bin"] = pd.cut(calibration.confidence, [0, .90, .93, .96, 1])
display(calibration.groupby("confidence_bin", observed=False)
        .agg(responses=("fully_correct", "size"), mean_confidence=("confidence", "mean"),
             share_fully_correct=("fully_correct", "mean")))

# Ranking: AUC = chance that a fully correct response has a higher confidence than one with an error
auc = roc_auc_score(calibration.fully_correct, calibration.confidence)
print(f"AUC of stated confidence for spotting fully correct responses: {auc:.2f} (0.5 = no better than chance)")
print(f"Mean confidence when fully correct: {calibration.loc[calibration.fully_correct, 'confidence'].mean():.3f}, "
      f"when at least one field is wrong: {calibration.loc[~calibration.fully_correct, 'confidence'].mean():.3f}")

Sample responses: 100
Mean stated confidence: 0.928
Share with all six fields correct: 0.550


,responses,mean_confidence,share_fully_correct
confidence_bin,,,
"(0.0, 0.9]",21,0.890,0.286
"(0.9, 0.93]",35,0.921,0.457
"(0.93, 0.96]",38,0.947,0.737
"(0.96, 1.0]",6,0.978,0.833


AUC of stated confidence for spotting fully correct responses: 0.73 (0.5 = no better than chance)
Mean confidence when fully correct: 0.937, when at least one field is wrong: 0.916


**Your answer (Q8):**

Over the 100 sample responses, the **mean stated confidence is 0.928**, but only **55% of the responses have all six fields right**. The model is clearly **over-confident**: it says "93% sure" and is fully right about half of the time.

| Stated confidence | Responses | Mean confidence | Share fully correct |
|---|---|---|---|
| up to 0.90 | 21 | 0.890 | 29% |
| 0.90 to 0.93 | 35 | 0.921 | 46% |
| 0.93 to 0.96 | 38 | 0.947 | 74% |
| above 0.96 | 6 | 0.978 | 83% |

Every group is right less often than it claims. Still, the share of fully correct answers rises steadily with the stated confidence, and the AUC is **0.73**: the confidence is **useful for ranking** (look at the least confident responses first), but it cannot be read as a probability. It is also a weak signal: fully correct responses average 0.937 and wrong ones 0.916, a gap too small to set a reliable threshold on. The agreement between five samples (Q6) is the cleaner signal.

## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [15]:
REVIEW_COST_EUR = 4      # a human checks one field
ERROR_COST_EUR = 150     # one error that reaches the lender's report

policies = {
    "P0: trust the vote": pd.Series(False, index=votes.index),
    "P1: review < 5/5": votes.n_agree < 5,
    "P2: review <= 3/5": votes.n_agree <= 3,
}


def policy_cost(sent_to_human):
    """The human fixes every field they review; the errors they never see stay in the report."""
    n_reviewed = int(sent_to_human.sum())
    errors_left = int((~votes.correct & ~sent_to_human).sum())
    return {"fields reviewed": n_reviewed, "errors left": errors_left,
            "review cost": REVIEW_COST_EUR * n_reviewed, "error cost": ERROR_COST_EUR * errors_left,
            "total cost (EUR)": REVIEW_COST_EUR * n_reviewed + ERROR_COST_EUR * errors_left}


policy_table = pd.DataFrame({name: policy_cost(rule) for name, rule in policies.items()}).T
display(policy_table)

best_policy = policy_table["total cost (EUR)"].astype(float).idxmin()
survivors = votes[~votes.correct & ~policies[best_policy]]
print(f"Cheapest: {best_policy}. Errors that survive it:")
display(survivors[["id", "field", "vote", "gold", "agreement"]])

# One extra check: send every number of an excerpt written in an unusual unit to a human, whatever the agreement.
# (A real list would come from a unit dictionary; these are units an English-trained model often leaves unconverted.)
UNUSUAL_UNITS = r"short ton|long ton|10,000 tonnes|lakh|crore"
unusual_unit_ids = excerpts.loc[excerpts.text.str.contains(UNUSUAL_UNITS, case=False), "id"].tolist()
unit_check = votes.id.isin(unusual_unit_ids) & votes.field.isin(NUMBER_FIELDS)
print(f"Excerpts flagged by the unit check: {unusual_unit_ids}")
policy_table.loc["P1 + unit check"] = policy_cost(policies["P1: review < 5/5"] | unit_check)
display(policy_table)

,fields reviewed,errors left,review cost,error cost,total cost (EUR)
P0: trust the vote,0,6,0,900,900
P1: review < 5/5,27,2,108,300,408
P2: review <= 3/5,12,3,48,450,498


Cheapest: P1: review < 5/5. Errors that survive it:


,id,field,vote,gold,agreement
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000",1.000
112,E19,production_mt,3.300,2.990,1.000


Excerpts flagged by the unit check: ['E03', 'E19']


,fields reviewed,errors left,review cost,error cost,total cost (EUR)
P0: trust the vote,0,6,0,900,900
P1: review < 5/5,27,2,108,300,408
P2: review <= 3/5,12,3,48,450,498
P1 + unit check,32,0,128,0,128


**Your answer (Q9):**

| Policy | Fields reviewed | Errors left | Review cost | Error cost | **Total** |
|---|---|---|---|---|---|
| P0: trust the vote | 0 | 6 | EUR 0 | EUR 900 | EUR 900 |
| P1: review < 5/5 | 27 | 2 | EUR 108 | EUR 300 | **EUR 408** |
| P2: review ≤ 3/5 | 12 | 3 | EUR 48 | EUR 450 | EUR 498 |

**P1 is the cheapest.** One error avoided (EUR 150) pays for 37 reviews (EUR 4 each), so it is worth reviewing generously. P1 beats P2 because its 15 extra reviews (EUR 60) also catch the E11 net zero error (2050 instead of 2053, agreement 4/5), which would have cost EUR 150.

The two errors that survive P1 are the unanimous ones from Q7: **E03 Scope 1 and E19 production**, both unit conversions. The one extra check we would add is a **unit rule**: if an excerpt uses an unusual unit (short tons, "10,000 tonnes", lakh...), all its numbers go to a human whatever the agreement. Here it adds 5 reviews and catches both errors: **EUR 128 in total, with no error left**. A plausibility check on intensity (Scope 1 / production between roughly 0.3 and 2.5 t per tonne of steel) would also flag E03 (0.19), but not E19 (1.55 looks normal). Honest caveat: we wrote our unit list after seeing these two errors, so in practice it has to come from a proper unit dictionary, and it will still miss units nobody listed.

---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = corpus["chunks"]
queries = pd.DataFrame(corpus["queries"])

vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(chunks)              # vocabulary and IDF learnt on the report
query_vectors = vectorizer.transform(queries["query"])        # words the report never uses are dropped
tfidf_similarity = cosine_similarity(query_vectors, chunk_vectors)   # one row per query, one column per chunk


def retrieval_table(similarity):
    """Top-3 chunks per query, and whether the relevant chunk is rank 1 (hit@1) or in the top 3 (hit@3)."""
    rows = []
    for query_number, query in queries.iterrows():
        ranking = np.argsort(-similarity[query_number], kind="stable")   # stable: ties keep chunk order
        top3 = ranking[:3].tolist()
        relevant = query["relevant"]
        rows.append({"qid": query["qid"], "query": query["query"], "relevant": relevant, "top3": top3,
                     "top1_score": similarity[query_number, top3[0]],
                     "hit@1": top3[0] in relevant,
                     "hit@3": any(chunk in relevant for chunk in top3)})
    return pd.DataFrame(rows).set_index("qid")


retrieval = retrieval_table(tfidf_similarity)
display(retrieval)

,query,relevant,top3,top1_score,hit@1,hit@3
qid,,,,,,
Q1,What were Scope 1 emissions in 2024?,[3],"[1, 5, 12]",0.338,False,False
Q2,net zero target year,[5],"[5, 0, 13]",0.366,True,True
Q3,crude steel production 2024,[2],"[2, 3, 0]",0.347,True,True
Q4,investment in hydrogen DRI capex,[6],"[6, 0, 1]",0.225,True,True
Q5,impact of the carbon price on EBITDA,[8],"[8, 13, 7]",0.346,True,True
Q6,flooding exposure of our plants,[9],"[3, 6, 0]",0.208,False,False


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [17]:
print(f"hit@1 = {retrieval['hit@1'].mean():.2f} ({retrieval['hit@1'].sum()} of {len(retrieval)}), "
      f"hit@3 = {retrieval['hit@3'].mean():.2f} ({retrieval['hit@3'].sum()} of {len(retrieval)})\n")

tokenize = vectorizer.build_analyzer()     # the same lower-casing, tokens and stop words as the vectorizer
for qid, row in retrieval[~retrieval["hit@1"]].iterrows():
    first_chunk, relevant_chunk = row["top3"][0], row["relevant"][0]
    query_words = set(tokenize(row["query"]))
    print(f"{qid}: '{row['query']}'   (top-1 score {row['top1_score']:.3f})")
    print(f"   retrieved first, chunk {first_chunk}: {chunks[first_chunk]}")
    print(f"   relevant,        chunk {relevant_chunk}: {chunks[relevant_chunk]}")
    print(f"   query words known to the report: {sorted(query_words & set(vectorizer.vocabulary_))}")
    print(f"   query words shared with the relevant chunk: {sorted(query_words & set(tokenize(chunks[relevant_chunk])))}")
    print(f"   query words shared with the retrieved chunk: {sorted(query_words & set(tokenize(chunks[first_chunk])))}\n")

hit@1 = 0.67 (4 of 6), hit@3 = 0.67 (4 of 6)



Q1: 'What were Scope 1 emissions in 2024?'   (top-1 score 0.338)
   retrieved first, chunk 1: About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 covers purchased electricity and heat, and Scope 3 covers our value chain.
   relevant,        chunk 3: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.
   query words known to the report: ['2024', 'emissions', 'scope']
   query words shared with the relevant chunk: ['2024', 'emissions']
   query words shared with the retrieved chunk: ['emissions', 'scope']

Q6: 'flooding exposure of our plants'   (top-1 score 0.208)
   retrieved first, chunk 3: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of 

**Your answer (Q10):**

(a) TF-IDF gives **hit@1 = 0.67 (4 of 6)** and **hit@3 = 0.67**: when it misses, the right chunk is not even in the top 3.

(b) The two failures:
* **Q1 "What were Scope 1 emissions in 2024?"** retrieves chunk 1, the "About this report" definitions, which says "Scope" three times. The relevant chunk 3 never says "Scope 1": it talks about "direct greenhouse gas emissions from our blast furnaces". On top of that, TF-IDF drops one-character tokens like "1", so for it "Scope 1", "Scope 2" and "Scope 3" are the same word. The whole top 3 (chunks 1, 5, 12) are chunks containing "scope".
* **Q6 "flooding exposure of our plants"** shares **no word at all** with the relevant chunk 9 ("riverine inundation", "most exposed", "sites"). TF-IDF only matches identical words: no synonyms (flooding / inundation, plants / sites) and no stemming (exposure / exposed). It returns chunk 3 only because "sinter plants" contains "plants", with a low score (0.21).

Both are vocabulary mismatches: the question and the report talk about the same thing in different words.

(c) Embeddings would most likely fix **Q6**: flooding and inundation, exposure and exposed are close in meaning, which is exactly what dense vectors capture. The optional D2 below confirms it. Q1 is harder, because "Scope 1 = direct emissions" is accounting jargon rather than plain meaning.

(d) An extraction error happens with the right text in front of the model: we can measure it, a reviewer who opens the source sees it, and resampling flags part of it. After a retrieval miss the model **never sees the answer**. Either it returns null, and the company silently looks like it does not report, or worse, it picks a plausible number from the wrong chunk: for Q1 the retrieved chunks contain "35% by 2030" and a Scope 3 of "7.8 MtCO2e", which could end up as Scope 1. All five samples see the same wrong context, so they agree and the answer looks certain, and a reviewer who checks it against the retrieved text finds it "supported". That is why retrieval has to be tested on its own (hit@k) before we trust the extraction.

> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [18]:
# Optional: dense embeddings. Needs `pip install sentence-transformers`; the model (about 90 MB) downloads on first use.
import logging
import os

# Keep the output readable: no download bars and no Hugging Face / torch start-up warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

try:
    from sentence_transformers import SentenceTransformer
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    SentenceTransformer = None
    print("sentence-transformers is not installed: D2 skipped.")

if SentenceTransformer is not None:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    # normalised vectors, so the cosine similarity is just the dot product
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    query_embeddings = embedder.encode(queries["query"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    dense_retrieval = retrieval_table(cosine_similarity(query_embeddings, chunk_embeddings))
    display(dense_retrieval)
    print(f"Dense:  hit@1 = {dense_retrieval['hit@1'].mean():.2f}, hit@3 = {dense_retrieval['hit@3'].mean():.2f}")
    print(f"TF-IDF: hit@1 = {retrieval['hit@1'].mean():.2f}, hit@3 = {retrieval['hit@3'].mean():.2f}")
    print("Still missed at rank 1 with embeddings:", dense_retrieval.index[~dense_retrieval["hit@1"]].tolist())

,query,relevant,top3,top1_score,hit@1,hit@3
qid,,,,,,
Q1,What were Scope 1 emissions in 2024?,[3],"[5, 12, 1]",0.577,False,False
Q2,net zero target year,[5],"[5, 0, 6]",0.595,True,True
Q3,crude steel production 2024,[2],"[2, 0, 3]",0.678,True,True
Q4,investment in hydrogen DRI capex,[6],"[6, 8, 7]",0.384,True,True
Q5,impact of the carbon price on EBITDA,[8],"[8, 13, 7]",0.791,True,True
Q6,flooding exposure of our plants,[9],"[9, 10, 3]",0.420,True,True


Dense:  hit@1 = 0.83, hit@3 = 0.83
TF-IDF: hit@1 = 0.67, hit@3 = 0.67
Still missed at rank 1 with embeddings: ['Q1']


**Our answer (D2, optional):**

Yes. With `all-MiniLM-L6-v2` embeddings, **hit@1 goes from 0.67 to 0.83 (5 of 6)**, and hit@3 is the same. The embeddings fix **Q6** (flooding / riverine inundation) and keep the four queries TF-IDF already found.

**Q1 still fails.** The model ranks chunk 5 ("reduce Scope 1 and 2 emissions by 35%"), chunk 12 (Scope 3) and chunk 1 (definitions) first, because they literally contain "Scope". Knowing that "Scope 1" means "direct emissions from blast furnaces" is accounting knowledge that a small general-purpose model does not have. Adding "direct emissions" to the query, or combining keyword and embedding search with a re-ranker, would be the next thing to try.

---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

> **To the Head of ESG Data.** We tested the AI tool on 20 steel-company report extracts. Asked once, it gets 87% of figures right (90% range: 78% to 93%), so reaching 95% is very unlikely. It sometimes invents missing figures and misreads unusual units such as short tons, repeating the same wrong number with full confidence. We recommend asking it five times and having an analyst check figures where answers differ or units are unusual: about EUR 6 per report. Limitation: our extracts were short and fictional; finding the right page in real reports adds errors.